In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------------------------
# 1. Cargar los datos
# -------------------------------------------------

datos = fetch_openml(
    name="adult",
    version=2,
    as_frame=True
)

X = datos.data
y = datos.target

# Seleccionar 10.000 observaciones
muestra = X.sample(n=10000, random_state=42).index
X = X.loc[muestra]
y = y.loc[muestra]

# -------------------------------------------------
# 2. Identificar tipos de variables
# -------------------------------------------------

variables_numericas = X.select_dtypes(
    include=["number"]
).columns.tolist()

variables_categoricas = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Transformar variables numericas
transformacion_numerica = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler())
])

# Transformar variables categoricas
transformacion_categorica = Pipeline([
    ("imputacion",
     SimpleImputer(strategy="most_frequent")),
    ("codificacion",
     OneHotEncoder(handle_unknown="ignore"))
])

# sparse_threshold=0 fuerza una matriz densa,
# necesaria para GaussianNB
preprocesamiento = ColumnTransformer(
    [
        (
            "numericas",
            transformacion_numerica,
            variables_numericas
        ),
        (
            "categoricas",
            transformacion_categorica,
            variables_categoricas
        )
    ],
    sparse_threshold=0
)

# -------------------------------------------------
# 3. Separar entrenamiento y prueba
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -------------------------------------------------
# 4. Crear y entrenar el clasificador
# -------------------------------------------------

modelo_bayes = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("clasificador", GaussianNB())
])

modelo_bayes.fit(X_train, y_train)

# -------------------------------------------------
# 5. Realizar las predicciones
# -------------------------------------------------

predicciones = modelo_bayes.predict(X_test)

print("Exactitud de Bayes ingenuo:",
      accuracy_score(y_test, predicciones))

print("\nReporte de clasificacion:")
print(classification_report(y_test, predicciones))

# -------------------------------------------------
# 6. Calcular probabilidades para un ejemplo
# -------------------------------------------------

ejemplo = X_test.iloc[[0]]

clase_predicha = modelo_bayes.predict(ejemplo)
probabilidades = modelo_bayes.predict_proba(ejemplo)

print("\nEjemplo seleccionado:")
print(ejemplo)

print("Clase predicha:", clase_predicha[0])

print("\nProbabilidades por clase:")
for clase, probabilidad in zip(
    modelo_bayes.classes_,
    probabilidades[0]
):
    print(clase, ":", round(probabilidad, 4))

Exactitud de Bayes ingenuo: 0.36

Reporte de clasificacion:
              precision    recall  f1-score   support

       <=50K       0.96      0.17      0.29      1532
        >50K       0.26      0.97      0.42       468

    accuracy                           0.36      2000
   macro avg       0.61      0.57      0.35      2000
weighted avg       0.79      0.36      0.32      2000


Ejemplo seleccionado:
       age workclass  fnlwgt education  education-num marital-status  \
15845   23   Private  349156   HS-grad              9  Never-married   

            occupation   relationship   race   sex  capital-gain  \
15845  Farming-fishing  Not-in-family  White  Male             0   

       capital-loss  hours-per-week native-country  
15845             0              45  United-States  
Clase predicha: >50K

Probabilidades por clase:
<=50K : 0.0
>50K : 1.0
